# My Custom Text-to-Image Lab
This is a little project I put together to play around with Stable Diffusion. Basically, you type in a prompt, and it spits out an image. I'm using the `diffusers` library because it's way easier than writing everything from scratch, but it still feels pretty powerful.

# ** Key Features**


### *   Uses a powerful pre-trained Stable Diffusion model.
### *   Accepts natural language prompts for image generation.
### *  Simple and minimal — runs with just a few lines of code.
### *  Uses diffusers, transformers, and torch for efficient performance.
### *  Integrated with Hugging Face login to access gated models.






# ** STEPS**

### Step 1: Getting the dependencies sorted
First things first, we need the right libraries. I'm grabbing `diffusers` for the model handling, `transformers` for the heavy lifting, and `accelerate` to make sure it doesn't take forever to run.

*Note: I added a little trick to restart the session after installing, just to make sure everything loads correctly.*

In [ ]:
!pip install -U diffusers transformers accelerate safetensors

import os

# restart the kernel so the new versions kick in
print("Done installing. Restarting now...")
os._exit(0)

In [ ]:
import diffusers
import transformers
import accelerate
import torch
print(f"Diffusers version: {diffusers.__version__}")
print(f"Is CUDA available: {torch.cuda.is_available()}")

In [ ]:

import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt

# 2. Logging into Hugging Face
The model we want to use (Stable Diffusion) is hosted on a platform called Hugging Face.
Think of it like a library full of AI models. To borrow some advanced models, we need to log in using our Hugging Face token.

Once we’re logged in, we can load powerful models that are otherwise restricted.

In [ ]:
from huggingface_hub import login
login("Your HF_TOKEN")

# 3. Loading the Stable Diffusion Model
Here’s where the magic begins! We load a pre-trained Stable Diffusion model using Hugging Face’s pipeline.
This model has already been trained on millions of images, so it understands how to generate new ones from text descriptions.

We also send it to the GPU (if available) using .to("cuda"), which makes it work much faster than on a regular CPU.

In [ ]:
from diffusers import DiffusionPipeline
import torch
import gc

# to clear out memory because these models are huge
def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("VRAM cleared.")

# Function to pull the main model
def get_base_model():
    print("--- Loading the SDXL Base Model ---")
    #  fp16 to save some space
    m = DiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    )

    # Only use cpu offload if we actually have a GPU to offload FROM
    if torch.cuda.is_available():
        m.enable_sequential_cpu_offload()
    return m

# Function to pull the refiner
def get_refiner_model():
    print("--- Loading the Refiner ---")
    m = DiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-refiner-1.0",
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True
    )
    if torch.cuda.is_available():
        m.enable_sequential_cpu_offload()
    return m

# 4. GIVING PROMPT

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# style options for the dropdown
style_options = [
    'None',
    'realistic image',
    'animated image',
    'sketch type image',
    'high vfx image'
]

# Create the dropdown widget
style_dropdown = widgets.Dropdown(
    options=style_options,
    value='None',
    description='Select Style:'
)

display(style_dropdown)

After selecting the style from the dropdown above, run the next cell to set your prompts.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Check if the dropdown exists in the current session
if 'style_dropdown' not in globals():
    print("Style dropdown was not found. Re-creating it now...")
    style_options = ['None', 'realistic image', 'animated image', 'sketch type image', 'high vfx image']
    style_dropdown = widgets.Dropdown(options=style_options, value='None', description='Select Style:')
    display(style_dropdown)

user_prompt = input("Enter a detailed prompt: ")
negative_prompt = input("Enter a negative prompt (e.g., 'blurry, ugly, deformed'): ")
style_prompt = style_dropdown.value

print(f"Selected Style: {style_prompt if style_prompt != 'None' else 'No specific style'}")


# 5. Then displays the generated image .

In [ ]:
import torch
import matplotlib.pyplot as plt

# Cleanup before we start
clear_vram()

#  Step 1: Prep the prompt
clean_prompt = user_prompt.replace('parket', 'parked')

# Add some 'magic' keywords if they want it realistic
if style_prompt == 'realistic image':
    my_final_prompt = f"{clean_prompt}. 8k uhd, highly detailed, professional photography, masterpiece"
else:
    my_final_prompt = f"{clean_prompt}, style: {style_prompt}"

print(f"Working on: {my_final_prompt}")

# Step 2: Generate the base latents
b_pipe = get_base_model()
b_pipe.safety_checker = None

with torch.no_grad():

    raw_latents = b_pipe(
        prompt=my_final_prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=30,
        denoising_end=0.8,
        output_type="latent",
    ).images

del b_pipe
clear_vram()

r_pipe = get_refiner_model()
r_pipe.vae.to(dtype=torch.float32)

with torch.no_grad():

    refined = r_pipe(
        prompt=my_final_prompt,
        num_inference_steps=30,
        denoising_start=0.8,
        image=raw_latents,
        output_type="latent",
    ).images

    # Decoding the result into an actual image
    print("Finalizing pixels...")
    refined = refined.to(dtype=torch.float32)
    final_latents = refined / r_pipe.vae.config.scaling_factor
    img_data = r_pipe.vae.decode(final_latents, return_dict=False)[0]
    final_img = r_pipe.image_processor.postprocess(img_data, output_type="pil")[0]

del r_pipe
clear_vram()

# Step 4: Look at the result
plt.figure(figsize=(10, 10))
plt.imshow(final_img)
plt.axis("off")
plt.title("My AI Creation")
plt.show()

## **Project Conclusion & Summary**

In this session, we transformed a basic Stable Diffusion v1.5 setup into a professional-grade **SDXL Base + Refiner** pipeline.

### **Key Accomplishments:**
1.  **High-Realism Upgrade:** Successfully transitioned to `stabilityai/stable-diffusion-xl-base-1.0` and the `refiner` model for superior image detail.
2.  **Memory Optimization:** Implemented a **Sequential Loading** strategy (Base -> Delete -> Refiner) to allow these massive models to run on a standard T4 GPU without crashing the session.
3.  **Numerical Stability Fix:** Resolved the persistent 'Black Image' bug common on T4 GPUs by forcing **Float32 precision** during the VAE decoding stage and disabling the safety checker.
4.  **User Experience:** Added a stylized UI using `ipywidgets` for easy selection of artistic styles like 'realistic' or 'vfx'.

This notebook is now optimized for generating high-quality AI art.
